# PaMCP MultiBenchmark v1.1 — Reproducible Colab runner

This notebook pins and records public benchmark snapshots, normalizes MCP-Atlas, MCPVerse and ToolRet with discovery/evaluator separation, and executes three ToolRet retrieval baselines (BM25, TF-IDF and a deterministic random sanity baseline). It reports macro task-level Precision/Recall/MRR/nDCG/Hit, paired task-bootstrap intervals, per-task metrics, plots and a ZIP artifact bundle.

**Scope boundary:** ToolRet is a heterogeneous retrieval track, not MCP execution. MCP-Atlas trajectories are not exhaustive tool-set ground truth. This notebook does not claim PaMCP superiority, semantic safety correctness, observed side-effect correctness, or full MCP server execution. Those require separate implementations and independent oracles.

In [ ]:
import os, subprocess, json, pathlib, hashlib, re, sys
REPO = 'https://github.com/msellamiTN/taas-provenance-discovery.git'
BRANCH = 'research/pamcp-multibenchmark-v1'
REPO_DIR = pathlib.Path('/content/taas-provenance-discovery')
if REPO_DIR.exists() and (REPO_DIR / '.git').exists():
    # IMPORTANT: an existing Colab checkout may contain an older adapter.
    # Always refresh it; merely checking that the directory exists is insufficient.
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','--depth','1','origin',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout','-B',BRANCH,'FETCH_HEAD'], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','FETCH_HEAD'], check=True)
else:
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} exists but is not a Git checkout; remove it or choose another path.')
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
commit = subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip()
print('Repository:', REPO, '\nBranch:', BRANCH, '\nCommit:', commit)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[experiment,benchmark-hf,test]'], check=True)


In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q',
 'tests/test_pamcp_benchmark_io.py',
 'tests/test_pamcp_benchmark_adapters.py',
 'tests/test_mcp_atlas_adapter.py',
 'tests/test_toolret_adapter.py',
 'tests/test_evaluation_metrics.py',
 'tests/test_declared_effects_are_not_observations.py'], check=True)

## 1. Acquire MCP-Atlas public dataset

The public release is 500 tasks. The full benchmark contains additional private tasks that are not acquired here. The dataset card declares CC-BY-4.0; preserve attribution and pin the exact dataset revision.

In [ ]:
# Colab recovery: make the checked-out src-layout package importable even if setup cell 1 was skipped.
from pathlib import Path
import sys, subprocess, os
_candidates = [Path.cwd(), Path('/content/taas-provenance-discovery')]
_repo_root = next((p.resolve() for p in _candidates if (p / 'src' / 'taas' / 'experiments').is_dir()), None)
if _repo_root is None:
    _repo_root = Path('/content/taas-provenance-discovery')
    if not _repo_root.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'research/pamcp-multibenchmark-v1',
                        'https://github.com/msellamiTN/taas-provenance-discovery.git', str(_repo_root)], check=True)
if not (_repo_root / 'src' / 'taas' / 'experiments').is_dir():
    raise RuntimeError(f"Repository source tree not found under {_repo_root}; rerun the setup cell and verify the checkout.")
os.chdir(_repo_root)
_src = str(_repo_root / 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)
# Editable installation supplies declared dependencies; sys.path fallback handles src-layout imports.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[experiment,benchmark-hf]'], check=True)
import taas
print('TaaS package:', Path(taas.__file__).resolve())

from huggingface_hub import HfApi, hf_hub_download
from taas.experiments.mcp_atlas_adapter import adapt_mcp_atlas_parquet
api = HfApi()
atlas_revision = os.environ.get('MCP_ATLAS_REVISION') or api.dataset_info('ScaleAI/MCP-Atlas').sha
atlas_info = api.dataset_info('ScaleAI/MCP-Atlas', revision=atlas_revision)
atlas_files = api.list_repo_files('ScaleAI/MCP-Atlas', repo_type='dataset', revision=atlas_revision)
parquet_files = [x for x in atlas_files if x.endswith('.parquet')]
assert len(parquet_files) == 1, f'Expected one public parquet file, found {parquet_files}'
atlas_path = hf_hub_download('ScaleAI/MCP-Atlas', parquet_files[0], repo_type='dataset', revision=atlas_revision)
atlas_manifest = adapt_mcp_atlas_parquet(atlas_path, 'data/processed/mcp_atlas_public')
atlas_manifest['source_revision'] = atlas_revision
pathlib.Path('data/processed/mcp_atlas_public/manifest.json').write_text(json.dumps(atlas_manifest, indent=2))
print(json.dumps(atlas_manifest['counts'], indent=2))

## 2b. Ingest MCPVerse time-invariant tasks

This adapter preserves the benchmark's annotated tool/server and native evaluation method in evaluator-only records. The CSV does not provide a complete tool catalogue, so it is not used as a standalone retrieval corpus.

In [ ]:
mcpverse_dir = pathlib.Path('/content/mcpverse')
MCPVERSE_REVISION = '995cc9eab953120c760b90d4e8e41ffa7fdc88f1'
if not mcpverse_dir.exists(): subprocess.run(['git','clone','https://github.com/hailsham/mcpverse.git',str(mcpverse_dir)], check=True)
subprocess.run(['git','-C',str(mcpverse_dir),'checkout',MCPVERSE_REVISION], check=True)
mcpverse_revision = subprocess.check_output(['git','-C',str(mcpverse_dir),'rev-parse','HEAD'], text=True).strip()
assert mcpverse_revision == MCPVERSE_REVISION
from taas.experiments.mcpverse_adapter import adapt_mcpverse_csv
mcpverse_csv = mcpverse_dir / 'mcpverse' / 'data' / 'mcpverse_time_invariant_v1.1.csv'
mcpverse_manifest = adapt_mcpverse_csv(mcpverse_csv, 'data/processed/mcpverse_time_invariant_v1_1')
mcpverse_manifest['source_revision'] = mcpverse_revision
pathlib.Path('data/processed/mcpverse_time_invariant_v1_1/manifest.json').write_text(json.dumps(mcpverse_manifest, indent=2))
print(json.dumps(mcpverse_manifest['counts'], indent=2))

## 2. Acquire ToolRet at independent immutable revisions

ToolRet stores queries and tools in separate Hugging Face repositories. Their revisions are pinned independently. The complete tool corpus is mandatory; deriving candidates from positive qrels alone would invalidate retrieval evaluation.

In [ ]:
import importlib, inspect
import taas.experiments.toolret_adapter as toolret_adapter
# Reload the module after repository synchronization. This prevents stale functions
# imported before the Git refresh from surviving in the Colab kernel.
importlib.invalidate_caches()
toolret_adapter = importlib.reload(toolret_adapter)
adapter_source = inspect.getsource(toolret_adapter.adapt_toolret_records)
assert 'Malformed ToolRet query record' in adapter_source, (
    'Stale ToolRet adapter is loaded. Restart the Colab runtime and run every cell from the beginning.'
)
query_revision = os.environ.get('TOOLRET_QUERY_REVISION') or api.dataset_info('mangopy/ToolRet-Queries').sha
tool_revision = os.environ.get('TOOLRET_TOOL_REVISION') or api.dataset_info('mangopy/ToolRet-Tools').sha
query_info = api.dataset_info('mangopy/ToolRet-Queries', revision=query_revision)
tool_info = api.dataset_info('mangopy/ToolRet-Tools', revision=tool_revision)
toolret_manifest = toolret_adapter.adapt_toolret_hf(
    'data/processed/toolret',
    query_revision=query_revision,
    tool_revision=tool_revision,
)
print('Adapter source:', toolret_adapter.__file__)
print('Repository commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print(json.dumps(toolret_manifest['counts'], indent=2))
print('Data quality:', json.dumps(toolret_manifest['data_quality'], indent=2))
print('Query revision:', query_info.sha)
print('Tool revision:', tool_info.sha)


## 3. Run ToolRet BM25 baseline

This is a heterogeneous retrieval baseline, not an MCP execution result. It uses the full ToolRet candidate corpus and original graded relevance judgments. The ranking metric evaluator penalizes abstentions as empty rankings.

## 3. ToolRet retrieval baselines and task-level evaluation

Compare BM25 and TF-IDF as lexical retrieval baselines, with deterministic random ranking as a sanity floor. All systems use the same complete candidate corpus, task prompts, graded qrels, cut-offs and evaluator. Per-task metrics are retained so uncertainty is computed over paired tasks rather than treating random seeds as independent observations. The bootstrap interval is descriptive and assumes independent tasks; use cluster resampling if shared task/server clusters are identified.

In [ ]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from taas.experiments.evaluation_metrics import ranking_metrics
import random, hashlib, csv, math

tools = json.loads(pathlib.Path('data/processed/toolret/discovery/tools.json').read_text())
tasks = json.loads(pathlib.Path('data/processed/toolret/discovery/tasks.json').read_text())
refs = json.loads(pathlib.Path('data/processed/toolret/evaluator/task_references.json').read_text())
qrels = {r['task_id']:{x['tool_id']:float(x.get('relevance',1)) for x in r['relevance_judgments']} for r in refs}
tool_ids = [t['tool_id'] for t in tools]
def tok(s): return re.findall(r'[a-z0-9_]+', str(s).lower())
tool_texts = [' '.join([t.get('name',''), t.get('description',''), json.dumps(t.get('input_schema',{}), sort_keys=True)]) for t in tools]
tool_docs = [tok(s) for s in tool_texts]
bm25 = BM25Okapi(tool_docs)
tfidf = TfidfVectorizer(tokenizer=tok, preprocessor=None, token_pattern=None, lowercase=False, norm='l2')
tool_matrix = tfidf.fit_transform(tool_texts)
systems = {'BM25': {}, 'TF-IDF': {}, 'Random (sanity)': {}}
per_task = {name: {} for name in systems}
for i, task in enumerate(tasks, start=1):
    task_id, query = task['task_id'], task['prompt']
    bm_scores = bm25.get_scores(tok(query))
    tf_scores = (tfidf.transform([query]) @ tool_matrix.T).toarray().ravel()
    bm_order = sorted(range(len(bm_scores)), key=lambda j: (-float(bm_scores[j]), tool_ids[j]))[:10]
    tf_order = sorted(range(len(tf_scores)), key=lambda j: (-float(tf_scores[j]), tool_ids[j]))[:10]
    seed = int(hashlib.sha256(task_id.encode('utf-8')).hexdigest()[:16], 16)
    shuffled = list(range(len(tool_ids))); random.Random(seed).shuffle(shuffled)
    systems['BM25'][task_id] = [tool_ids[j] for j in bm_order]
    systems['TF-IDF'][task_id] = [tool_ids[j] for j in tf_order]
    systems['Random (sanity)'][task_id] = [tool_ids[j] for j in shuffled[:10]]
    if qrels.get(task_id):
        for system_name, rankings in systems.items():
            one = ranking_metrics({task_id: rankings[task_id]}, {task_id: qrels[task_id]}, ks=(1,5,10))
            per_task[system_name][task_id] = {metric: one['metrics'][str(k)][f'{metric}@{k}']['value']
                for k in (1,5,10) for metric in ('precision','recall','mrr','ndcg','hit')}
    if i % 500 == 0: print(f'Ranked {i}/{len(tasks)} tasks')

aggregate = {name: ranking_metrics(rankings, qrels, ks=(1,5,10)) for name, rankings in systems.items()}
out = pathlib.Path('artifacts/multibenchmark/toolret_baselines'); out.mkdir(parents=True, exist_ok=True)
for name, rankings in systems.items():
    safe_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('-', '_')
    (out/f'{safe_name}_rankings.json').write_text(json.dumps(rankings, indent=2))
(out/'aggregate_metrics.json').write_text(json.dumps(aggregate, indent=2))
(out/'per_task_metrics.json').write_text(json.dumps(per_task, indent=2))

# Paired task bootstrap for descriptive uncertainty; tasks are the resampling unit.
# If multiple tasks share a common source/server/request cluster, replace this with cluster resampling.
def paired_bootstrap(left, right, metric, n_boot=10000, seed=20261105):
    ids = sorted(set(left) & set(right))
    vals = [float(right[t][metric]) - float(left[t][metric]) for t in ids]
    if not vals: return {'estimate': None, 'ci95': [None, None], 'n_tasks': 0, 'unit': 'task'}
    rng = random.Random(seed); draws = sorted(sum(rng.choices(vals, k=len(vals)))/len(vals) for _ in range(n_boot))
    lo, hi = int(0.025*n_boot), min(n_boot-1, int(0.975*n_boot))
    return {'estimate': sum(vals)/len(vals), 'ci95': [draws[lo], draws[hi]], 'n_tasks': len(vals), 'unit': 'task', 'resamples': n_boot, 'seed': seed}
comparisons = {}
for competitor in ('TF-IDF', 'Random (sanity)'):
    comparisons[f'BM25_minus_{competitor}'] = {m: paired_bootstrap(per_task['BM25'], per_task[competitor], m)
        for m in ('ndcg@5','hit@5','recall@5','mrr@5')}
(out/'paired_task_bootstrap.json').write_text(json.dumps(comparisons, indent=2))

with (out/'per_task_metrics.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['system','task_id','metric','value']); writer.writeheader()
    for system_name, rows in per_task.items():
        for task_id, vals in rows.items():
            for metric, value in vals.items(): writer.writerow({'system':system_name,'task_id':task_id,'metric':metric,'value':value})
print(json.dumps({'aggregate': aggregate, 'paired_bootstrap': comparisons}, indent=2))

In [ ]:
# Figures: baseline comparison, metric profile and benchmark candidate-pool diagnostics.
import matplotlib.pyplot as plt
plot_dir = pathlib.Path('artifacts/multibenchmark/plots'); plot_dir.mkdir(parents=True, exist_ok=True)
ks = [1,5,10]
plt.figure(figsize=(8,5))
for name, result in aggregate.items():
    vals = [result['metrics'][str(k)][f'ndcg@{k}']['value'] for k in ks]
    plt.plot(ks, vals, marker='o', linewidth=2, label=name)
plt.xlabel('Cut-off k'); plt.ylabel('Macro nDCG@k'); plt.title('ToolRet — retrieval baselines'); plt.xticks(ks); plt.grid(True, alpha=.25); plt.legend(); plt.tight_layout()
plt.savefig(plot_dir/'toolret_ndcg_comparison.png', dpi=220); plt.show()

plt.figure(figsize=(8,5))
for name, result in aggregate.items():
    vals = [result['metrics'][str(k)][f'hit@{k}']['value'] for k in ks]
    plt.plot(ks, vals, marker='s', linewidth=2, label=name)
plt.xlabel('Cut-off k'); plt.ylabel('Macro Hit@k'); plt.title('ToolRet — hit-rate comparison'); plt.xticks(ks); plt.grid(True, alpha=.25); plt.legend(); plt.tight_layout()
plt.savefig(plot_dir/'toolret_hit_comparison.png', dpi=220); plt.show()

k = 5; metric_names = ['precision','recall','mrr','ndcg','hit']
x = list(range(len(metric_names))); width = 0.24
plt.figure(figsize=(9,5))
for offset, (name, result) in enumerate(aggregate.items()):
    vals = [result['metrics'][str(k)][f'{m}@{k}']['value'] for m in metric_names]
    plt.bar([v + (offset-1)*width for v in x], vals, width=width, label=name)
plt.xticks(x, [f'{m}@{k}' for m in metric_names]); plt.ylabel('Macro task-level score'); plt.title('ToolRet — metric profile at k=5'); plt.grid(axis='y', alpha=.25); plt.legend(); plt.tight_layout()
plt.savefig(plot_dir/'toolret_metric_profile_k5.png', dpi=220); plt.show()

pool_sizes = [len(x['enabled_tool_ids']) for x in json.loads(pathlib.Path('data/processed/mcp_atlas_public/discovery/task_tool_visibility.json').read_text())]
plt.figure(figsize=(8,5)); plt.hist(pool_sizes, bins=range(min(pool_sizes), max(pool_sizes)+2), edgecolor='black')
plt.xlabel('Enabled tools per MCP-Atlas task'); plt.ylabel('Task count'); plt.title('MCP-Atlas public release — candidate-pool distribution'); plt.grid(axis='y', alpha=.2); plt.tight_layout()
plt.savefig(plot_dir/'mcp_atlas_candidate_pool.png', dpi=220); plt.show()
print('Saved plots to', plot_dir.resolve())

## 4. Reproducibility and export

Keep the raw source revisions, generated manifests, per-task rankings, metrics and plots together. Do not publish only aggregate scores. The MCP-Atlas claims and trajectories remain evaluator-side; this notebook does not convert the successful trajectory into an exhaustive set of acceptable toolchains.

In [ ]:
# Export raw per-task outputs, aggregate metrics, source revisions, figures and environment metadata.
import zipfile, platform, datetime
artifact_root = pathlib.Path('artifacts/multibenchmark'); artifact_root.mkdir(parents=True, exist_ok=True)
summary = {
  'repository_commit': subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip(),
  'python_version': sys.version,
  'platform': platform.platform(),
  'run_timestamp_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
  'mcp_atlas_revision': atlas_revision,
  'mcpverse_revision': mcpverse_revision,
  'toolret_query_revision': query_revision,
  'toolret_tool_revision': tool_revision,
  'toolret_baselines': aggregate,
  'paired_task_bootstrap': comparisons,
  'scientific_status': 'ToolRet BM25, TF-IDF and deterministic random sanity baseline executed; MCP-Atlas and MCPVerse ingested; no PaMCP superiority claim; E3 effect oracle not supplied',
  'limitations': ['ToolRet is a heterogeneous retrieval track, not MCP execution.', 'Task-level bootstrap assumes tasks are independent; use cluster resampling if task clusters are identified.', 'MCP-Atlas trajectories are not exhaustive acceptable-tool-set ground truth.', 'No semantic safety oracle or observed-effects oracle is provided by this notebook.']
}
(artifact_root/'run_summary.json').write_text(json.dumps(summary, indent=2))
zip_path = '/content/PaMCP_MultiBenchmark_v1_artifacts.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
  for p in pathlib.Path('data/processed').rglob('*'):
    if p.is_file(): z.write(p, p.relative_to('.'))
  for p in artifact_root.rglob('*'):
    if p.is_file(): z.write(p, p.relative_to('.'))
print('Artifact bundle:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Not running in Google Colab; retrieve the ZIP from /content manually.')